# Lab 4: Mini Transformer NMT

In [1]:
import math
import random
from pathlib import Path

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


## TODO: Implement Mini Transformer

## 1) Build Vocabulary and Encode Sentences

In [3]:
SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]
PAD, BOS, EOS, UNK = SPECIALS


def tokenize(text):
    return text.strip().lower().split()


def build_vocab(sentences, min_freq=2):
    freq = {}
    for s in sentences:
        for tok in tokenize(s):
            freq[tok] = freq.get(tok, 0) + 1

    itos = SPECIALS.copy()
    for tok, c in sorted(freq.items(), key=lambda x: (-x[1], x[0])):
        if c >= min_freq:
            itos.append(tok)

    stoi = {tok: i for i, tok in enumerate(itos)}
    return stoi, itos


def encode_sentence(sentence, stoi):
    ids = [stoi[BOS]]
    ids.extend(stoi.get(tok, stoi[UNK]) for tok in tokenize(sentence))
    ids.append(stoi[EOS])
    return torch.tensor(ids, dtype=torch.long)


def resolve_or_download_data_dir():
    candidate_dirs = [
        Path("data"),
        Path("NLP Lab 4_ Mini Transformer NMT") / "data",
        Path("Seq2Seq-Attention") / "NLP Lab 4_ Mini Transformer NMT" / "data",
        Path("/content") / "data",
        Path("/content") / "Seq2Seq-Attention" / "NLP Lab 4_ Mini Transformer NMT" / "data",
    ]

    required_files = ["train.en", "train.vi", "dev.en", "dev.vi"]

    for d in candidate_dirs:
        if all((d / f).exists() for f in required_files):
            return d

    data_dir = Path("data")
    data_dir.mkdir(parents=True, exist_ok=True)

    from urllib.request import urlretrieve

    base_url = (
        "https://raw.githubusercontent.com/dangnt-courses/hands-on-projects/main/"
        "Seq2Seq-Attention/NLP%20Lab%204_%20Mini%20Transformer%20NMT/data"
    )

    for f in required_files:
        target = data_dir / f
        if not target.exists():
            urlretrieve(f"{base_url}/{f}", str(target))

    if all((data_dir / f).exists() for f in required_files):
        return data_dir

    raise FileNotFoundError(
        "Could not locate Lab 4 data files. Expected: train.en, train.vi, dev.en, dev.vi."
    )


base_dir = resolve_or_download_data_dir()
train_en_lines = (base_dir / "train.en").read_text(encoding="utf-8").splitlines()
train_vi_lines = (base_dir / "train.vi").read_text(encoding="utf-8").splitlines()

max_samples = 8000
pairs = list(zip(train_en_lines, train_vi_lines))[:max_samples]

src_stoi, src_itos = build_vocab([x[0] for x in pairs], min_freq=2)
tgt_stoi, tgt_itos = build_vocab([x[1] for x in pairs], min_freq=2)

src_pad_idx = src_stoi[PAD]
tgt_pad_idx = tgt_stoi[PAD]


dataset = [(encode_sentence(src, src_stoi), encode_sentence(tgt, tgt_stoi)) for src, tgt in pairs]
split = int(0.9 * len(dataset))
train_data = dataset[:split]
valid_data = dataset[split:]


def collate_fn(batch):
    src_batch = [x[0] for x in batch]
    tgt_batch = [x[1] for x in batch]
    src_pad = pad_sequence(src_batch, batch_first=True, padding_value=src_pad_idx)
    tgt_pad = pad_sequence(tgt_batch, batch_first=True, padding_value=tgt_pad_idx)
    return src_pad.to(device), tgt_pad.to(device)


train_loader = DataLoader(train_data, batch_size=64, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_data, batch_size=64, shuffle=False, collate_fn=collate_fn)

print(f"Data dir: {base_dir}")
print(f"Train pairs: {len(train_data)} | Valid pairs: {len(valid_data)}")
print(f"Src vocab: {len(src_itos)} | Tgt vocab: {len(tgt_itos)}")

Data dir: data
Train pairs: 9 | Valid pairs: 1
Src vocab: 11 | Tgt vocab: 13


## 2) Positional Encoding and Mini Transformer

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        position = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]

        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)


class MiniTransformerNMT(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=256,
        nhead=8,
        num_encoder_layers=3,
        num_decoder_layers=3,
        dim_feedforward=512,
        dropout=0.1,
    ):
        super().__init__()
        self.d_model = d_model

        self.src_embed = nn.Embedding(src_vocab_size, d_model, padding_idx=src_pad_idx)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model, padding_idx=tgt_pad_idx)
        self.pos_enc = PositionalEncoding(d_model=d_model, dropout=dropout)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )

        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def make_src_padding_mask(self, src):
        return src.eq(src_pad_idx)

    def make_tgt_padding_mask(self, tgt):
        return tgt.eq(tgt_pad_idx)

    def make_tgt_subsequent_mask(self, tgt_len, device):
        return torch.triu(torch.ones(tgt_len, tgt_len, device=device, dtype=torch.bool), diagonal=1)

    def forward(self, src, tgt_input):
        src_key_padding_mask = self.make_src_padding_mask(src)
        tgt_key_padding_mask = self.make_tgt_padding_mask(tgt_input)
        tgt_mask = self.make_tgt_subsequent_mask(tgt_input.size(1), tgt_input.device)

        src_emb = self.pos_enc(self.src_embed(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_enc(self.tgt_embed(tgt_input) * math.sqrt(self.d_model))

        output = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )

        logits = self.fc_out(output)
        return logits

## 3) Train the Transformer

In [5]:
model = MiniTransformerNMT(
    src_vocab_size=len(src_itos),
    tgt_vocab_size=len(tgt_itos),
    d_model=256,
    nhead=8,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=512,
    dropout=0.1,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=tgt_pad_idx)


def train_epoch(model, dataloader):
    model.train()
    total_loss = 0.0

    for src, tgt in dataloader:
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        optimizer.zero_grad()
        logits = model(src, tgt_input)

        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / max(len(dataloader), 1)


@torch.no_grad()
def evaluate(model, dataloader):
    model.eval()
    total_loss = 0.0

    for src, tgt in dataloader:
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        logits = model(src, tgt_input)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))
        total_loss += loss.item()

    return total_loss / max(len(dataloader), 1)


epochs = 4
for epoch in range(1, epochs + 1):
    train_loss = train_epoch(model, train_loader)
    valid_loss = evaluate(model, valid_loader)
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Valid Loss: {valid_loss:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 01 | Train Loss: 2.8931 | Valid Loss: 2.3330
Epoch 02 | Train Loss: 2.5270 | Valid Loss: 2.3158
Epoch 03 | Train Loss: 2.2884 | Valid Loss: 1.7945
Epoch 04 | Train Loss: 1.7954 | Valid Loss: 1.7426


## 4) Greedy Decoding

In [6]:
@torch.no_grad()
def greedy_decode(model, src, max_len=40):
    model.eval()

    batch_size = src.size(0)
    ys = torch.full((batch_size, 1), tgt_stoi[BOS], dtype=torch.long, device=src.device)

    for _ in range(max_len):
        logits = model(src, ys)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        ys = torch.cat([ys, next_token], dim=1)

    return ys


def decode_ids(ids, itos):
    tokens = []
    for idx in ids:
        tok = itos[int(idx)]
        if tok == EOS:
            break
        if tok not in {BOS, PAD}:
            tokens.append(tok)
    return " ".join(tokens)


sample_src = "i love natural language processing"
src_tensor = encode_sentence(sample_src, src_stoi).unsqueeze(0).to(device)
pred_ids = greedy_decode(model, src_tensor, max_len=40)[0].tolist()
pred_text = decode_ids(pred_ids[1:], tgt_itos)

print("SRC :", sample_src)
print("PRED:", pred_text)

SRC : i love natural language processing
PRED: <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk> <unk>
